> ⚠️ **作業中 (Work in Progress)**: このノートブックは現在開発中です。一部のコードが不完全であったり変更される可能性があります。

## 📋 目次

- [評価概要](#評価概要)
- [環境設定](#環境-設定)
- [評価基準の理解](#評価基準の理解)
- [評価の実行と結果分析](#評価の実行と結果分析)
- [評価のベストプラクティス](#評価のベストプラクティス)

## 🎯 学習目標

- AIエージェント評価の重要性の理解
- Foundryの自動評価機能の活用
- 様々な評価指標の意味と活用方法の学習
- 合成データを使用した評価の実行
- 評価結果の解釈と改善方法の導出

## ⏱️ 予想所要時間

約10分

## 評価概要

### なぜ 評価が 重要したが?

AI エージェントを 本番環境に デプロイすること 前に 次の 事項を 検証する必要があり します:

```
正確性 → 関連性 → 一貫性 → 自然さ → 安全性
```

評価 ないが デプロイすると:
- ❌ 不正確な 回答で ユーザー 信頼 低下
- ❌ 関連 ないは レスポンスで ユーザー 経験 悪化
- ❌ 一貫性 ないは 品質で ブランド が未知 損傷
- ❌ 不適切な コンテンツ 作成で 法的 問題

### Microsoft Foundryの 評価 機能

Foundryは 次のを 自動化します:
- ✅ テスト データ 作成 (Synthetic generation)
- ✅ 様々な 評価 指標 適用
- ✅ 大規模 評価 実行
- ✅ 結果 可視化 および 分析

## 環境設定

評価を ための 環境を 設定します.

In [ ]:
# 環境変数 ロード
import json
import os
import subprocess

# PATH 環境変数の設定 (Azure CLIを 見つけられるように)
possible_paths = [
    "/opt/homebrew/bin",  # macOS (Apple Silicon)
    "/usr/local/bin",     # macOS (Intel) / Linux
    "/usr/bin",           # Linux / GitHub Codespaces
    "/home/linuxbrew/.linuxbrew/bin"  # Linux Homebrew
]

az_path = None
try:
    result = subprocess.run(['which', 'az'], capture_output=True, text=True)
    if result.returncode == 0:
        az_path = os.path.dirname(result.stdout.strip())
except:
    pass

paths_to_add = []
if az_path and az_path not in os.environ.get("PATH", ""):
    paths_to_add.append(az_path)
else:
    for path in possible_paths:
        if os.path.exists(path) and path not in os.environ.get("PATH", ""):
            paths_to_add.append(path)

if paths_to_add:
    new_path = ":".join(paths_to_add) + ":" + os.environ.get("PATH", "")
    os.environ["PATH"] = new_path

# が前 ノートブックで 保存した設定ファイルのロード
config_file = ".foundry_config.json"
try:
    with open(config_file, 'r') as f:
        config = json.load(f)
    
    # 環境変数 設定
    FOUNDRY_NAME = config.get("FOUNDRY_NAME")
    RESOURCE_GROUP = config.get("RESOURCE_GROUP")
    LOCATION = config.get("LOCATION")
    TENANT_ID = config.get("TENANT_ID")
    PROJECT_NAME = config.get("PROJECT_NAME", "proj-default")
    PROJECT_ENDPOINT = config.get("FOUNDRY_ENDPOINT")
    
    # 環境変数でも 設定 (他のツールが使用できるように)
    os.environ["FOUNDRY_NAME"] = FOUNDRY_NAME
    os.environ["LOCATION"] = LOCATION
    os.environ["RESOURCE_GROUP"] = RESOURCE_GROUP
    os.environ["AZURE_SUBSCRIPTION_ID"] = config.get("AZURE_SUBSCRIPTION_ID", "")
    os.environ["_ENDPOINT"] = PROJECT_ENDPOINT
    
    print(f"✅ 設定ファイル '{config_file}'で 環境変数を ロードしました.")
    print(f"\n📌 Foundry Name: {FOUNDRY_NAME}")
    print(f"📌 Resource Group: {RESOURCE_GROUP}")
    print(f"📌 Location: {LOCATION}")
    print(f"📌 プロジェクト エンドポイント: {PROJECT_ENDPOINT}")
    
except FileNotFoundError:
    print(f"⚠️ '{config_file}' ファイルを 見つかりません.")
    print("💡 01-setup.ipynbを 先に実行して環境を 設定してください.")
    raise

# 必須 パッケージ インストール
%pip install -q azure-ai-evaluation azure-ai-projects azure-identity

from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

print(f"\n💡 使用するプロジェクトエンドポイント: {PROJECT_ENDPOINT}")


In [ ]:
# 簡単なエージェント評価 例
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

# テストデータの準備
test_queries = [
    "Pythonで リストをソートする方法は?",
    "機械学習とディープラーニングの違いを説明してください.",
    "ソウルの人口は何人ですか?",
    "最近のAI技術の動向を教えてください.",
    "クラウド コンピューティングのメリットは何ですか?"
]

print("📊 エージェント テスト 開始...\n")
print("=" * 80)

# 自動で ModelRouterAgent 検索
try:
    agents = list(project_client.agents.list())
    agent_router = next((a for a in agents if a.name == "ModelRouterAgent"), None)
    
    if not agent_router:
        print("⚠️ 'ModelRouterAgent'を 見つかりません.")
        print("💡 03-agents.ipynbを 先に実行してエージェントを作成してください.")
        raise ValueError("Agent not found")
    
    print(f"✅ 使用する エージェント: {agent_router.name} (ID: {agent_router.id})\n")
    
    # デモ用 Mock レスポンス (実際 環境では Portal 評価 推奨)
    print("\n💡 参考: SDK バージョン 制限で により Mock レスポンスを 使用します.")
    print("   実際 大規模 評価は Azure Portalを 使用してください.\n")
    
    mock_responses = [
        "Pythonで リストを ソートするには `sort()` メソッドや `sorted()` 関数を 使用する できる あります. `list.sort()`は リストを その場で ソートして, `sorted(list)`は 新しいで運 ソートされた リストを 返却します.",
        "機械学習は 明示的 プログラミング ないが データで パターンを 学習するは 技術がであり, ディープラーニングは 人工 ニューラルネットワークを 使用するは 機械学習の した 分野です. ディープラーニングは より 複雑な パターンを 学習できる できる ありだけ より 多いは データと コンピューティング パワーが 必要です.",
        "ソウルの 人口は 約 950だけ 名です. (2023年 基準)",
        "最近 AI 技術 動向では 大規模 言語 モデル(LLM)の 発展, マルチモーダル AI, 作成型 AIの 拡散, AI エージェント システムの 発展 などが あります.",
        "クラウド コンピューティングの 主要 メリットは 拡張性, コスト 効率性, アクセス性, 自動 更新, 災害 復旧 機能 などです. 必要に に従って リソースを 柔軟に 調整する できる あって 効率的です."
    ]
    
    responses = []
    for i, query in enumerate(test_queries, 1):
        print(f"\n[質問 {i}/{len(test_queries)}]: {query}")
        response = mock_responses[i-1]
        responses.append({"query": query, "response": response})
        print(f"[レスポンス]: {response[:100]}...")
    
    print("\n" + "=" * 80)
    print(f"\n✅ {len(test_queries)}個の質問テスト完了!")
    print(f"\n💡 ポータルでより詳細な評価を行ってください:")
    print("   https://ai.azure.com > Build > Evaluations")

except Exception as e:
    print(f"\n⚠️ エラー 発生: {e}")
    print("\n💡 解決方法:")
    print("   1. 03-agents.ipynbを まず 実行して ModelRouterAgentを 作成")
    print("   2. Azureに ログインしているか確認 (az login)")
    print("   3. FOUNDRY_NAMEが 正しいか 確認")

### Azure AI Evaluation SDK 使用すること

SDKを通じて プログラミング 方式で 評価を 実行する できる あります. 様々な Evaluatorを 使用して エージェント レスポンスの 品質を 測定します.

In [ ]:
# Azure AI Evaluation SDK Evaluator インポート
from azure.ai.evaluation import (
    CoherenceEvaluator,
    FluencyEvaluator,
    GroundednessEvaluator,
    RelevanceEvaluator,
)

# Azure OpenAI モデル 設定 (評価用)
model_config = {
    "azure_endpoint": f"https://{FOUNDRY_NAME}.openai.azure.com/",
    "api_version": "2024-08-01-preview",
    "azure_deployment": "gpt-5.1",  # 評価に 使用する モデル (するがオープン 含む)
}

# Evaluator インスタンス 作成
coherence_evaluator = CoherenceEvaluator(model_config=model_config)
fluency_evaluator = FluencyEvaluator(model_config=model_config)
groundedness_evaluator = GroundednessEvaluator(model_config=model_config)
relevance_evaluator = RelevanceEvaluator(model_config=model_config)

print("✅ Evaluator インスタンスが 作成なりました:")
print("   - CoherenceEvaluator: 論理的 一貫性 評価")
print("   - FluencyEvaluator: 自然さ 評価")
print("   - GroundednessEvaluator: 事実 ベース 評価")
print("   - RelevanceEvaluator: 関連性 評価")

In [ ]:
# 評価 実行 例
# 上で 収集した responses データを 使用して 評価します

print("📊 SDK ベース 評価 実行 中...\n")
print("=" * 80)

evaluation_results = []

for i, item in enumerate(responses, 1):
    query = item["query"]
    response = item["response"]
    
    # 評価用 コンテキスト (実際 環境では RAGで が取得した ドキュメント 使用)
    context = "一般的な ナレッジベース 質問に に対する レスポンスです."
    
    print(f"\n[サンプル {i}/{len(responses)}]")
    print(f"質問: {query[:50]}...")
    
    # 各 Evaluatorで 評価
    coherence_score = coherence_evaluator(query=query, response=response)
    fluency_score = fluency_evaluator(query=query, response=response)
    groundedness_score = groundedness_evaluator(query=query, response=response, context=context)
    relevance_score = relevance_evaluator(query=query, response=response, context=context)
    
    result = {
        "query": query,
        "response": response[:100],
        "coherence": coherence_score.get("coherence", "N/A"),
        "fluency": fluency_score.get("fluency", "N/A"),
        "groundedness": groundedness_score.get("groundedness", "N/A"),
        "relevance": relevance_score.get("relevance", "N/A"),
    }
    evaluation_results.append(result)
    
    print(f"  Coherence: {result['coherence']}/5")
    print(f"  Fluency: {result['fluency']}/5")
    print(f"  Groundedness: {result['groundedness']}/5")
    print(f"  Relevance: {result['relevance']}/5")

print("\n" + "=" * 80)
print(f"\n✅ {len(responses)}個 サンプル 評価 完了!")

In [ ]:
# 評価 結果 概要
import statistics

print("=" * 80)
print("📈 評価 結果 概要")
print("=" * 80)

# 各 指標別 平均 計算
metrics = ["coherence", "fluency", "groundedness", "relevance"]
averages = {}

for metric in metrics:
    scores = [r[metric] for r in evaluation_results if isinstance(r[metric], (int, float))]
    if scores:
        averages[metric] = statistics.mean(scores)
    else:
        averages[metric] = "N/A"

print(f"\n📊 Overall Scores:")
print(f"   Coherence:    {averages['coherence']:.2f}/5.0" if isinstance(averages['coherence'], float) else f"   Coherence:    {averages['coherence']}")
print(f"   Fluency:      {averages['fluency']:.2f}/5.0" if isinstance(averages['fluency'], float) else f"   Fluency:      {averages['fluency']}")
print(f"   Groundedness: {averages['groundedness']:.2f}/5.0" if isinstance(averages['groundedness'], float) else f"   Groundedness: {averages['groundedness']}")
print(f"   Relevance:    {averages['relevance']:.2f}/5.0" if isinstance(averages['relevance'], float) else f"   Relevance:    {averages['relevance']}")

# Pass/Fail 基準 (4.0 が上が場合 Pass)
threshold = 3.5
passed = sum(1 for r in evaluation_results 
             if all(isinstance(r[m], (int, float)) and r[m] >= threshold for m in metrics))
pass_rate = (passed / len(evaluation_results)) * 100 if evaluation_results else 0

print(f"\n✅ Pass Rate: {pass_rate:.1f}% ({passed}/{len(evaluation_results)} samples)")
print(f"   (基準: すべての 指標 ≥ {threshold})")

print("\n" + "=" * 80)
print("\n💡 推奨 事項:")
if averages.get('groundedness', 5) < 4.0:
    print("   ⚠️ Groundedness 改善 必要: Knowledge Base 補強 または Instructions 修正")
if averages.get('relevance', 5) < 4.0:
    print("   ⚠️ Relevance 改善 必要: エージェント Instructionsを より 明確に")
if averages.get('coherence', 5) < 4.0:
    print("   ⚠️ Coherence 改善 必要: レスポンス 構造化 ガイドライン 追加")
if averages.get('fluency', 5) < 4.0:
    print("   ⚠️ Fluency 改善 必要: モデル アップグレがする または プロンプト 改善")
if pass_rate >= 80:
    print("   ✅ 全体的で 良好な パフォーマンスです!")

# 結果 可視化 (選択事項)
try:
    import matplotlib.pyplot as plt
    
    # 指標別 平均 スコア 棒 グラフ
    valid_metrics = {k: v for k, v in averages.items() if isinstance(v, (int, float))}
    
    if valid_metrics:
        plt.figure(figsize=(10, 6))
        plt.bar(valid_metrics.keys(), valid_metrics.values())
        plt.axhline(y=threshold, color='r', linestyle='--', label=f'Threshold ({threshold})')
        plt.ylim(0, 5)
        plt.ylabel('Score')
        plt.title('Evaluation Results Summary')
        plt.legend()
        plt.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        print("\n📊 可視化 完了!")
except ImportError:
    print("\n💡 可視化を ために matplotlib インストール: pip install matplotlib")

### Foundry 提供 Evaluator 全体 リスト

Foundryは 6個 カテゴリ, 32個の Evaluatorを 提供します:

| カテゴリ | Evaluator 例 |
|---------|---------------|
| **一般 品質** | CoherenceEvaluator, FluencyEvaluator, QAEvaluator |
| **テキスト 類似も** | SimilarityEvaluator, F1ScoreEvaluator, BleuScoreEvaluator |
| **RAG** | GroundednessEvaluator, RelevanceEvaluator, RetrievalEvaluator |
| **エージェント** | IntentResolutionEvaluator, TaskAdherenceEvaluator, ToolCallAccuracyEvaluator |
| **リスク/安全** | ViolenceEvaluator, SexualEvaluator, ContentSafetyEvaluator |
| **Azure OpenAI Graders** | AzureOpenAILabelGrader, AzureOpenAIGrader |

詳細な 内容は [Azure AI Evaluation ドキュメント](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/evaluation-evaluators)を 参照してください.

## 評価 基準 がして

Foundryは 次の 4がない コア 評価 基準を 提供します:

| 基準 | 説明 |
|------|------|
| **Groundedness** | レスポンスが 提供された コンテキスト/知識に ベースするはない |
| **Relevance** | レスポンスが 質問と 関連が あるか |
| **Coherence** | レスポンスが 論理的で 一貫性 あるか |
| **Fluency** | レスポンスが 自然で 文法的で 正しいか |

# 簡単なエージェント評価 例
from azure.ai.projects import AIProjectClient
from azure.identity import DefaultAzureCredential

credential = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)

# テストデータの準備
test_queries = [
    "Pythonで リストをソートする方法は?",
    "機械学習とディープラーニングの違いを説明してください.",
    "ソウルの人口は何人ですか?",
    "最近のAI技術の動向を教えてください.",
    "クラウド コンピューティングのメリットは何ですか?"
]

print("📊 エージェント テスト 開始...\n")
print("=" * 80)

# エージェント レスポンス 収集
agent_id = "<your-agent-id>"  # ⚠️ 03-agentsで 作成した エージェント ID

responses = []
for i, query in enumerate(test_queries, 1):
    print(f"\n[質問 {i}/{len(test_queries)}]: {query}")
    
    # Thread 作成
    thread = project_client.agents.create_thread()
    
    # メッセージ 送信
    message = project_client.agents.create_message(
        thread_id=thread.id,
        role="user",
        content=query
    )
    
    # Run 実行
    run = project_client.agents.create_and_process_run(
        thread_id=thread.id,
        assistant_id=agent_id
    )
    
    # レスポンス 収集
    messages = project_client.agents.list_messages(thread_id=thread.id)
    response = messages.data[0].content[0].text.value
    
    responses.append({"query": query, "response": response})
    print(f"[レスポンス]: {response[:100]}...")
    
print("\n" + "=" * 80)
print(f"\n✅ {len(test_queries)}個の質問テスト完了!")
print(f"\n💡 ポータルでより詳細な評価を行ってください:")
print("   https://ai.azure.com > Build > Evaluations")

### 各 基準が 重要した が有

| Groundedness | Relevance | Coherence | Fluency |
|--------------|-----------|-----------|----------|
| ユーザー 信頼 確保 | ユーザー だけ足も 向上 | が理解する 簡単な 回答 | ユーザー 経験 向上 |
| 法的 責任 最小化 | 効率的 情報 伝達 | 専門的な が未知 | ブランド が未知 維持 |
| 虚偽 情報 防止 | 会話 フロー 維持 | 信頼性 向上 | がしても 増が |

## 評価 実行 および 結果 分析

### 評価 実行

1. **Submit** ボタン クリック
2. 評価が バックグラウンドで 実行なります (約 10-15分)
3. Evaluations ページで 進行 状況 確認

### 結果 解釈

### 大規模 評価は Portal 使用 推奨

**SDK vs Portal 比較:**

| 機能 | Python SDK | Azure Portal |
|------|-----------|--------------|
| サンプル できる | 小規模 (10-20) | 大規模 (50-200+) |
| 可視化 | 制限的 | 豊富な ダッシュボード |
| Synthetic データ | 手動 作成 | 自動 作成 |




















| **Human Evaluation** | 自動 評価と 並行して 新しいで運 問題 パターン 発見 || **ベースライン スコア** | Groundedness ≥4.0 / 残り ≥3.5 / Pass rate ≥80% || **評価 周期** | 開発 中: 毎 更新 / デプロイ 前: 必須 / デプロイ 後: 週間/月間 || **テスト シナリオ** | 一般·複雑·曖昧·多言語 質問 + Edge cases || **サンプル できる** | 開発: 10-20個 / テスト: 50-100個 / 本番環境: 200+個 ||------|----------|| 項目 | 推奨 事項 |### 推奨 事項6. Submit → 結果 確認 (10-15分 所要)5. Criteria: Groundedness, Relevance, Coherence, Fluency 選択4. Data: Synthetic generation (50-200 サンプル)3. Target: Agent 選択 (はい: ModelRouterAgent)2. + Create new evaluation クリック1. https://ai.azure.com → Build → Evaluations**Portalで 評価する方法:**| 協業 | 困難 | チーム 共有 可能 || 結果 管理 | でカラム 保存 | クラウド 保存 および 共有 |Pass Rate: 85% (43/50 samples)
```

**改善が 必要な 領域:**
- Groundednessが 4.0 がハイン サンプル レビュー
- 失敗した 7個 サンプル 分析
- Instructions 改善 または Knowledge Base 補強

## 📚 追加リソース

- [Azure AI Evaluation 概要](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/observability?view=foundry#what-are-evaluators)
- [Foundry ポータルで 評価 実行](https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/evaluate-generative-ai-app?view=foundry)
- [エージェント 評価](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/evaluation-evaluators/agent-evaluators?view=foundry)

## 📚 追加リソース

- [Azure AI Evaluation 概要](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/observability?view=foundry#what-are-evaluators)
- [Foundry ポータルで 評価 実行](https://learn.microsoft.com/en-us/azure/ai-foundry/how-to/evaluate-generative-ai-app?view=foundry)
- [エージェント 評価](https://learn.microsoft.com/en-us/azure/ai-foundry/concepts/evaluation-evaluators/agent-evaluators?view=foundry)